# Lecture 6
## Model selection:
* Bias variance decomposition and model stability
* Regularization: Ridge regression

In [ ]:
"""
Generate n random data points following y = -0.5*x^2 - 0.5
Gaussian noise added to y, and save them to a CSV file.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N = 30
X_MIN, X_MAX = -5, 5
NOISE_STD = 0.5
SEED = 0
CSV_PATH = "data.csv"


def generate_data(n=N, x_min=X_MIN, x_max=X_MAX, noise_std=NOISE_STD, seed=SEED):
    rng = np.random.default_rng(seed)
    x = rng.uniform(x_min, x_max, n)
    y = -0.5 * x**2 - 0.5 + rng.normal(0, noise_std, n)
    return x, y


def main():
    x, y = generate_data()

    df = pd.DataFrame({"x": x, "y": y})
    df.to_csv(CSV_PATH, index=False)
    print(f"Saved {N} points to {CSV_PATH}")

    order = np.argsort(x)
    x_true = np.linspace(X_MIN, X_MAX, 300)
    y_true = -0.5 * x_true**2 - 0.5

    plt.figure(figsize=(6.5, 4.5))
    plt.scatter(x, y, alpha=0.6, label="generated data")
    plt.plot(x_true, y_true, color="red", label=r"$y = -0.5x^2 - 0.5$")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.tight_layout()

if __name__ == "__main__":
    main()

## Bias and variance - Model stability

In [ ]:
"""
Generate three independent samples drawn from the same generating
process, y = -0.5*x^2 - 0.5 + noise, all using the same x-range. A
separate degree-GRADE polynomial regression is fit to each sample
independently (three separate models, not one model evaluated three
times), to demonstrate the STABILITY of the fitted model across
different random samples from the same distribution.

This is a direct, visual illustration of the variance term in the
bias-variance decomposition: if the three independently fitted curves
are nearly identical, the model is stable (low variance); if they
diverge noticeably from one another despite being fit on data from
the exact same generator, the model is unstable (high variance) -
typically the case for higher polynomial degrees.

Bias^2 and variance are also estimated numerically, in the usual way:
by repeating the sample-and-fit procedure many times and observing,
at each of a grid of test points, how far the average prediction is
from the true function (bias) and how much predictions vary across
the repeated fits (variance):

    bias^2(x0)   = ( mean_over_trials[ y_hat(x0) ] - y_true(x0) )^2
    variance(x0) =  variance_over_trials[ y_hat(x0) ]

averaged over the grid. The three plotted samples/fits are a small,
directly visible instance of exactly this repeated-sampling idea (n=3
instead of N_TRIALS), which is why the same bias^2/variance numbers
are reported alongside them.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N = 200
NOISE_STD = 0.5
SEED = 0

X_MIN, X_MAX = -5, 5   # same x-range used for all three samples

GRADE = 1          # degree g of the polynomial regression fit
N_TRIALS = 200      # number of resampled data sets used to estimate bias/variance


def true_function(x):
    return -0.5 * x**2 - 0.5


def generate_data(n, x_min, x_max, noise_std=NOISE_STD, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    x = rng.uniform(x_min, x_max, n)
    y = true_function(x) + rng.normal(0, noise_std, n)
    return x, y


def fit_polynomial_regression(x, y, grade):
    """Least-squares polynomial fit via numpy's built-in polyfit."""
    coeffs = np.polyfit(x, y, grade)
    return np.poly1d(coeffs)


def estimate_bias_variance(grade, n_trials=N_TRIALS, n_points=N,
                            x_min=X_MIN, x_max=X_MAX, noise_std=NOISE_STD):
    """Repeat the sample-and-fit procedure n_trials times, evaluate
    every fit on a fixed grid of test points, and compute the average
    squared bias and average variance across that grid."""
    rng = np.random.default_rng(SEED + 1)  # separate stream from the plotted samples
    x_grid = np.linspace(x_min, x_max, 100)
    y_true_grid = true_function(x_grid)

    predictions = np.zeros((n_trials, len(x_grid)))
    for t in range(n_trials):
        x_sample, y_sample = generate_data(n_points, x_min, x_max, noise_std, rng)
        poly = fit_polynomial_regression(x_sample, y_sample, grade)
        predictions[t] = poly(x_grid)

    mean_pred = predictions.mean(axis=0)
    bias_sq = np.mean((mean_pred - y_true_grid) ** 2)
    variance = np.mean(predictions.var(axis=0))

    return bias_sq, variance


def plot_sample(ax, x, y, poly, title, mse):
    x_smooth = np.linspace(X_MIN, X_MAX, 300)

    ax.scatter(x, y, alpha=0.5, label="data")
    ax.plot(x_smooth, true_function(x_smooth), color="red", label="true function")
    ax.plot(x_smooth, poly(x_smooth), color="green", linestyle="--",
            label=f"fitted model (grade {GRADE})")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper center")
    ax.text(0.03, 0.03, f"MSE = {mse:.3f}", transform=ax.transAxes, fontsize=8,
            verticalalignment="bottom",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))


def plot_overlay(ax, polys, colors):
    """Overlay all three independently fitted curves on the same axes,
    with no data points, to directly show how much they agree."""
    x_smooth = np.linspace(X_MIN, X_MAX, 300)
    ax.plot(x_smooth, true_function(x_smooth), color="black", linewidth=2,
            label="true function")
    for i, (poly, color) in enumerate(zip(polys, colors), start=1):
        ax.plot(x_smooth, poly(x_smooth), color=color, linestyle="--",
                label=f"fit on sample {i}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Overlay: all three fitted models (stability check)")
    ax.legend(fontsize=8)


def main():
    rngs = [np.random.default_rng(SEED + i) for i in range(3)]
    samples = [generate_data(N, X_MIN, X_MAX, rng=rng) for rng in rngs]

    for i, (x, y) in enumerate(samples, start=1):
        pd.DataFrame({"x": x, "y": y}).to_csv(f"sample{i}.csv", index=False)
    print("Saved sample1.csv, sample2.csv, sample3.csv")

    # Fit a separate model to each sample.
    polys = [fit_polynomial_regression(x, y, GRADE) for x, y in samples]
    mses = [np.mean((poly(x) - y) ** 2) for poly, (x, y) in zip(polys, samples)]

    for i, (poly, mse) in enumerate(zip(polys, mses), start=1):
        print(f"Sample {i} fit (grade {GRADE}): {poly}  ->  MSE = {mse:.4f}")

    # Bias/variance of the model class (degree GRADE, sample size N,
    # x-range as above) - a property of the estimator, estimated from
    # many more resamples than just the three plotted here.
    bias_sq, variance = estimate_bias_variance(GRADE)
    print(f"Estimated bias^2 (grade {GRADE}):  {bias_sq:.4f}")
    print(f"Estimated variance (grade {GRADE}): {variance:.4f}")
    print(f"Irreducible noise variance:         {NOISE_STD**2:.4f}")

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    colors = ["green", "purple", "orange"]

    for i, ((x, y), poly, mse) in enumerate(zip(samples, polys, mses)):
        row, col = divmod(i, 2)
        plot_sample(axes[row][col], x, y, poly, f"Sample {i+1}: independent fit", mse)

    plot_overlay(axes[1][1], polys, colors)

    fig.suptitle(
        f"Stability of grade-{GRADE} polynomial regression across three "
        f"independent samples\n(bias$^2$={bias_sq:.3f}, variance={variance:.3f}, "
        f"noise var.={NOISE_STD**2:.3f})"
    )
    fig.tight_layout()
    fig.savefig("stability_demo.png", dpi=150)
    print("Saved plot to stability_demo.png")


if __name__ == "__main__":
    main()

## Ridge regression

In [ ]:
"""
Generate a single training sample drawn from y = log(x) + noise, and
fit three L2-regularized (ridge) linear regressions to that SAME data,
using three different regularization strengths LAMBDA. A separate
HOLD-OUT set, drawn from the same generating process but over a wider
x-range that extends beyond the training data's boundaries, is used to
evaluate how well each of the three fitted models generalizes -
particularly to the extrapolated region the models never saw during
training.

Model:  y_hat = theta_0 + theta_1*x + theta_2*x^2 + ... + theta_g*x^g
Loss:   ridge objective,
            L(theta) = (1/n)*sum((y_hat_i - y_i)^2) + lambda * ||theta_{1:}||_2^2
        (the intercept theta_0 is excluded from the penalty, as is
        conventional, since penalizing it has no regularizing effect
        and would just bias predictions toward zero)

Fitting uses scikit-learn's Ridge (inside a Pipeline that also
standardizes the polynomial features, so the penalty falls evenly
across degrees and the fit stays numerically well-conditioned even at
high GRADE), not a hand-derived closed form.

For each lambda, bias^2 and variance are also estimated in the usual
way: by repeating the sample-and-fit procedure many times (fresh
training data each time, same lambda) and observing, at each of a grid
of HOLD-OUT-RANGE test points, how far the average prediction is from
the true function (bias) and how much predictions vary across the
repeated fits (variance). Evaluating this on the wider hold-out range
rather than just the training range shows both parts of the usual
regularization trade-off at once: how much lambda increases in-range
bias, and how much lambda controls (or fails to control) the
model's tendency to diverge when extrapolating.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

N = 200
NOISE_STD = 0.3
SEED = 0

TRAIN_X_MIN, TRAIN_X_MAX = 1, 10     # log(x) requires x > 0
TEST_X_MIN, TEST_X_MAX = 0.2, 20     # hold-out set extends beyond the training range
N_TEST = 100

GRADE = 8                    # degree g of the polynomial features
LAMBDAS = [0.0, 1.0, 50.0]   # three regularization strengths to compare
N_TRIALS = 200                # number of resampled data sets used to estimate bias/variance


def true_function(x):
    return np.log(x)


def generate_data(n, x_min, x_max, noise_std=NOISE_STD, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    x = rng.uniform(x_min, x_max, n)
    y = true_function(x) + rng.normal(0, noise_std, n)
    return x, y


def fit_ridge_regression(x, y, grade, lam):
    """L2-regularized polynomial regression via scikit-learn's Ridge,
    fit on polynomial features [x, x^2, ..., x^grade]. The features
    are standardized before Ridge is applied: raw polynomial features
    of different degrees live on very different scales (e.g. x vs.
    x^8), and the ridge penalty treats all coefficients equally, so
    without standardizing, the penalty would fall very unevenly across
    degrees (and, at high degree, the raw features become numerically
    ill-conditioned). The intercept is handled separately by Ridge and
    is not penalized."""
    pipeline = make_pipeline(
        PolynomialFeatures(degree=grade, include_bias=False),
        StandardScaler(),
        Ridge(alpha=lam),
    )
    pipeline.fit(x.reshape(-1, 1), y)
    return pipeline


def predict(pipeline, x):
    return pipeline.predict(x.reshape(-1, 1))


def estimate_bias_variance(grade, lam, n_trials=N_TRIALS, n_points=N,
                            x_min=TRAIN_X_MIN, x_max=TRAIN_X_MAX,
                            eval_min=TEST_X_MIN, eval_max=TEST_X_MAX,
                            noise_std=NOISE_STD):
    """Repeat the sample-and-fit procedure n_trials times (fresh
    training data each time, drawn from [x_min, x_max], fixed lambda),
    evaluate every fit on a fixed grid of test points spanning the
    (wider) hold-out range [eval_min, eval_max], and compute the
    average squared bias and average variance across that grid."""
    rng = np.random.default_rng(SEED + 1)  # separate stream from the plotted sample
    x_grid = np.linspace(eval_min, eval_max, 150)
    y_true_grid = true_function(x_grid)

    predictions = np.zeros((n_trials, len(x_grid)))
    for t in range(n_trials):
        x_sample, y_sample = generate_data(n_points, x_min, x_max, noise_std, rng)
        pipeline = fit_ridge_regression(x_sample, y_sample, grade, lam)
        predictions[t] = predict(pipeline, x_grid)

    mean_pred = predictions.mean(axis=0)
    bias_sq = np.mean((mean_pred - y_true_grid) ** 2)
    variance = np.mean(predictions.var(axis=0))

    return bias_sq, variance


def plot_fit(ax, x_train, y_train, x_test, y_test, pipeline, lam, title,
             train_mse, test_mse, bias_sq, variance):
    x_smooth = np.linspace(TEST_X_MIN, TEST_X_MAX, 300)

    ax.scatter(x_train, y_train, alpha=0.5, color="tab:blue", label="training data")
    ax.scatter(x_test, y_test, alpha=0.5, color="tab:red", marker="x", label="hold-out data")
    ax.axvspan(TRAIN_X_MIN, TRAIN_X_MAX, color="grey", alpha=0.1, label="training range")
    ax.plot(x_smooth, true_function(x_smooth), color="black", label="true function")
    ax.plot(x_smooth, predict(pipeline, x_smooth), color="green",
            linestyle="--", label=f"ridge fit ($\\lambda$={lam})")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.legend(fontsize=7, loc="lower right")

    textstr = (f"train MSE = {train_mse:.3f}\n"
               f"hold-out MSE = {test_mse:.3f}\n"
               f"bias$^2$ (hold-out) = {bias_sq:.3f}\n"
               f"variance (hold-out) = {variance:.3f}")
    ax.text(0.03, 0.97, textstr, transform=ax.transAxes, fontsize=8,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))


def plot_overlay(ax, x_train, y_train, x_test, y_test, fits, lambdas, colors):
    """Overlay all three fitted curves (same training data, different
    lambda) on the same axes, evaluated across the wider hold-out
    range, to directly show how well each generalizes/extrapolates."""
    x_smooth = np.linspace(TEST_X_MIN, TEST_X_MAX, 300)
    ax.scatter(x_train, y_train, alpha=0.3, color="tab:blue", label="training data")
    ax.scatter(x_test, y_test, alpha=0.3, color="tab:red", marker="x", label="hold-out data")
    ax.axvspan(TRAIN_X_MIN, TRAIN_X_MAX, color="grey", alpha=0.1, label="training range")
    ax.plot(x_smooth, true_function(x_smooth), color="black", linewidth=2,
            label="true function")
    for pipeline, lam, color in zip(fits, lambdas, colors):
        ax.plot(x_smooth, predict(pipeline, x_smooth), color=color,
                linestyle="--", label=f"$\\lambda$={lam}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Overlay: extrapolation beyond the training range")
    ax.legend(fontsize=7)


def main():
    rng_train = np.random.default_rng(SEED)
    rng_test = np.random.default_rng(SEED + 500)

    x_train, y_train = generate_data(N, TRAIN_X_MIN, TRAIN_X_MAX, rng=rng_train)
    x_test, y_test = generate_data(N_TEST, TEST_X_MIN, TEST_X_MAX, rng=rng_test)

    pd.DataFrame({"x": x_train, "y": y_train}).to_csv("train_data.csv", index=False)
    pd.DataFrame({"x": x_test, "y": y_test}).to_csv("holdout_data.csv", index=False)
    print("Saved train_data.csv, holdout_data.csv")

    # Fit one model per lambda, all on the same training data.
    fits = [fit_ridge_regression(x_train, y_train, GRADE, lam) for lam in LAMBDAS]
    train_mses = [np.mean((predict(pipeline, x_train) - y_train) ** 2) for pipeline in fits]
    test_mses = [np.mean((predict(pipeline, x_test) - y_test) ** 2) for pipeline in fits]

    bias_variance = []
    for lam in LAMBDAS:
        bias_sq, variance = estimate_bias_variance(GRADE, lam)
        bias_variance.append((bias_sq, variance))

    for lam, pipeline, train_mse, test_mse, (bias_sq, variance) in zip(
        LAMBDAS, fits, train_mses, test_mses, bias_variance
    ):
        ridge = pipeline.named_steps["ridge"]
        print(f"lambda={lam:>6}: train MSE={train_mse:.4f}  hold-out MSE={test_mse:.4f}  "
              f"bias^2(hold-out)={bias_sq:.4f}  variance(hold-out)={variance:.4f}  "
              f"max|coef|={np.max(np.abs(ridge.coef_)):.2f}")
    print(f"Irreducible noise variance: {NOISE_STD**2:.4f}")

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    colors = ["green", "purple", "orange"]

    for i, (lam, pipeline, train_mse, test_mse, (bias_sq, variance)) in enumerate(
        zip(LAMBDAS, fits, train_mses, test_mses, bias_variance)
    ):
        row, col = divmod(i, 2)
        plot_fit(axes[row][col], x_train, y_train, x_test, y_test, pipeline, lam,
                 f"$\\lambda$={lam}", train_mse, test_mse, bias_sq, variance)

    plot_overlay(axes[1][1], x_train, y_train, x_test, y_test, fits, LAMBDAS, colors)

    fig.suptitle(
        f"Effect of $\\lambda$ on a grade-{GRADE} ridge fit, evaluated on a hold-out "
        f"set extending beyond the training range [{TRAIN_X_MIN}, {TRAIN_X_MAX}]"
    )
    fig.tight_layout()
    fig.savefig("ridge_lambda_comparison.png", dpi=150)
    print("Saved plot to ridge_lambda_comparison.png")


if __name__ == "__main__":
    main()